# Python Random Data Generation Exercises: 10 Coding Problems with Solutions

A practice notebook on the `random` and `secrets` modules — random numbers, unique sampling, secure OTPs/tokens, constrained password generation, seeding, and random dates — each with a concept note, a hint, a solution, and an explanation.

*Adapted for practice from the exercise list at [PYnative](https://pynative.com/python-random-number-generation-exercise-questions-and-challenge/). Exercises 2 and 7 had bugs on the source page (confirmed by multiple reader comments) — Exercise 2's original solution didn't actually guarantee unique lottery tickets, and Exercise 7 printed the wrong variable. Both are fixed here.*

---

## Concepts you'll need

This set covers Python's **`random`** module (fast, general-purpose randomness) and **`secrets`** module (cryptographically secure randomness).

- **`random` module basics** — `random.random()` (float in [0.0, 1.0)), `random.uniform(a, b)` (float in a range), `random.randrange(start, stop, step)` (integer from a range, with an optional step), `random.randint(a, b)` (integer, inclusive on both ends), `random.choice(seq)` (one random element), `random.sample(population, k)` (k **unique** elements, no repeats), `random.shuffle(list)` (shuffles in place).
- **Uniqueness matters** — `random.sample()` guarantees no duplicates among its own picks, but calling `randrange()` repeatedly in a loop does **not** guarantee the results themselves are unique across calls — a common bug. Generating unique values directly with `random.sample(range(...), k)`, or accumulating into a `set()` until it reaches the target size, are the two correct fixes.
- **`random.seed(n)`** — resets the generator to a reproducible state; calling it with the same seed right before each `random` call makes that call's "random" output deterministic and repeatable — useful for testing, not for genuine randomness.
- **`secrets` module** — for anything security-sensitive (OTPs, tokens, passwords), prefer `secrets` over `random`, since `random`'s underlying generator (Mersenne Twister) is predictable and not safe for security purposes. Key functions: `secrets.randbelow(n)`, `secrets.choice(seq)`, `secrets.SystemRandom()` (an object with the same interface as the `random` module, backed by the OS's secure entropy source), `secrets.token_hex(n)` and `secrets.token_urlsafe(n)` (n random bytes, encoded as text).
- **Building constrained random strings** — combine `string.ascii_letters`/`digits`/`punctuation` as a character pool with `random.choice()` in a loop or comprehension; for passwords with *minimum* character-type requirements, generate the required characters explicitly, fill the rest randomly, then `shuffle()` the combined result so the guaranteed characters aren't predictably placed.
- **Random dates** — the cleanest modern approach converts both boundary dates to `datetime.date` objects, computes the day span with `(end - start).days`, picks a random offset in that range with `random.randint(0, span)`, and adds it back with `timedelta(days=offset)` — no manual Unix-timestamp math needed.

Each exercise below gives a problem, a hint, a solution, and an explanation.

## Exercise 1. Generate 3 Random Integers

**Concept:** random.randrange() with a step argument

**Problem:** Generate 3 random integers between 100 and 999 that are each divisible by 5.

**Given:**
```
range 100-999, divisible by 5
```

**Expected Output:**
```
e.g. 105, 730, 995 (values vary by run)
```

**Hint:** randrange(start, stop, step) only ever lands on values reachable by stepping from start — step=5 guarantees divisibility.

In [ ]:
import random

print("Generating 3 random integers between 100 and 999 divisible by 5")
for num in range(3):
    print(random.randrange(100, 999, 5), end=', ')

**Explanation:** randrange(100, 999, 5) only ever produces values from the sequence 100, 105, 110, ..., since the step parameter restricts which values are reachable at all — this guarantees divisibility by 5 without any extra filtering logic. The upper bound (999) is exclusive, matching range()'s usual behavior.

## Exercise 2. Random Lottery Pick

**Concept:** generating guaranteed-unique random values with random.sample() over a range

**Problem:** Generate 100 unique 10-digit lottery ticket numbers, then pick 2 lucky winners.

**Given:**
```
100 tickets, each 10 digits long, all unique
```

**Expected Output:**
```
2 winning ticket numbers, guaranteed distinct from each other and drawn from 100 guaranteed-unique tickets
```

**Hint:** Calling randrange() repeatedly in a loop does NOT guarantee unique results — random.sample(range(...), k) does.

In [ ]:
import random

# random.sample() draws k unique values directly from the range - no manual
# duplicate-checking loop needed, and no risk of silently repeated tickets.
lottery_tickets_list = random.sample(range(1_000_000_000, 10_000_000_000), 100)
print(f"Created {len(lottery_tickets_list)} unique 10-digit lottery tickets")
print(f"All unique: {len(set(lottery_tickets_list)) == len(lottery_tickets_list)}")

winners = random.sample(lottery_tickets_list, 2)
print("Lucky 2 lottery tickets are", winners)

**Explanation:** A common bug here is calling random.randrange() in a loop 100 times and assuming the results are all different — they aren't guaranteed to be, since each call is independent and duplicates can and do occur. random.sample(population, k) instead is specifically designed to return k UNIQUE elements with no repeats, so sampling 100 values directly from the full range(1_000_000_000, 10_000_000_000) guarantees every ticket is distinct in one call. The upper bound is 10_000_000_000 (not 9_999_999_999) since range()'s stop value is exclusive, and 9999999999 needs to be a reachable ticket number too.

## Exercise 3. Generate 6-digit Random Secure OTP

**Concept:** secrets.SystemRandom() for cryptographically secure ranges

**Problem:** Generate a secure 6-digit one-time password.

**Given:**
```
range 100000-999999
```

**Expected Output:**
```
A 6-digit OTP, e.g. 483920 (varies by run)
```

**Hint:** secrets.SystemRandom() has the same interface as the random module but is backed by the OS's secure entropy source.

In [ ]:
import secrets

secretsGenerator = secrets.SystemRandom()

print("Generating 6 digit random OTP")
otp = secretsGenerator.randrange(100000, 999999)

print("Secure random OTP is", otp)

**Explanation:** secrets.SystemRandom() is a class with the exact same method interface as the random module (.randrange(), .choice(), etc.), but its underlying randomness is drawn from the operating system's cryptographically secure entropy source rather than the predictable Mersenne Twister algorithm random uses internally. For anything security-sensitive — OTPs, tokens, session IDs — this predictability difference is exactly why secrets is the correct module, not random.

## Exercise 4. Pick Random Character

**Concept:** random.choice() on a string

**Problem:** Select one random character from a given string.

**Given:**
```
name = "pynative"
```

**Expected Output:**
```
A single character, e.g. 'y' (varies by run)
```

**Hint:** Strings are sequences in Python, so random.choice() works on them directly, just like on a list.

In [ ]:
import random

name = 'pynative'
char = random.choice(name)
print("random char is", char)

**Explanation:** random.choice(seq) picks one element uniformly at random from any sequence — since strings are iterable sequences of characters in Python, passing one directly works with no need to first convert it to a list of characters.

## Exercise 5. Generate Random String

**Concept:** random.choice() combined with a comprehension, or random.choices()

**Problem:** Generate a random 5-character string using only uppercase and lowercase letters, no digits or symbols.

**Given:**
```
length = 5, letters only
```

**Expected Output:**
```
A 5-letter string, e.g. 'xQaZm' (varies by run)
```

**Hint:** string.ascii_letters already contains both cases combined — no need to build the character pool manually.

In [ ]:
import random
import string

def randomString(stringLength):
    """Generate a random string of the given length, letters only."""
    letters = string.ascii_letters
    return ''.join(random.choice(letters) for i in range(stringLength))

print("Random String is", randomString(5))

# Equivalent, more concise alternative using random.choices() (plural)
print("Random String (choices()):", "".join(random.choices(string.ascii_letters, k=5)))

**Explanation:** string.ascii_letters is a ready-made constant containing all 52 upper- and lowercase letters combined, so no manual character-set assembly is needed. The comprehension calls random.choice() once per position, and "".join(...) glues the individually-chosen characters into one string. random.choices() (plural, with an 's') is a more concise built-in alternative that does the same 'pick k characters with replacement' operation directly, returning a list that still needs joining.

## Exercise 6. Generate Random Password

**Concept:** guaranteeing minimum character-type counts, then shuffling to avoid a predictable pattern

**Problem:** Generate a 10-character password with at least 2 uppercase letters, 1 digit, and 1 special symbol.

**Given:**
```
length = 10, minimum 2 upper, 1 digit, 1 symbol
```

**Expected Output:**
```
A 10-character password meeting all the minimum requirements, e.g. 'K9!mXq2R@p' (varies by run)
```

**Hint:** Generate the guaranteed characters explicitly first, fill the rest randomly, then shuffle — otherwise the guaranteed characters would predictably cluster at the start.

In [ ]:
import random
import string

def randomPassword():
    randomSource = string.ascii_letters + string.digits + string.punctuation

    # Fill the remaining length with fully random characters
    password = random.sample(randomSource, 6)
    # Guarantee the minimum required character types
    password += random.sample(string.ascii_uppercase, 2)
    password += random.choice(string.digits)
    password += random.choice(string.punctuation)

    passwordList = list(password)
    random.SystemRandom().shuffle(passwordList)
    password = ''.join(passwordList)
    return password

print("Password is", randomPassword())

**Explanation:** Simply picking 10 fully random characters from a combined pool cannot GUARANTEE the minimum counts requested — it's statistically likely but not certain. This solution instead explicitly generates the required minimum characters (2 uppercase, 1 digit, 1 symbol) plus 6 more general characters to reach length 10, then shuffles the combined list so the guaranteed characters don't predictably cluster at fixed positions, which would itself be a security weakness. random.SystemRandom().shuffle() uses the more secure random source for this final shuffle step, appropriate since this is password-generation code.

## Exercise 7. Calculate Multiplication

**Concept:** random.random() vs. random.uniform() for different float ranges

**Problem:** Multiply two random floats from two different ranges.

**Given:**
```
first float in [0.1, 1), second float in [9.5, 99.5]
```

**Expected Output:**
```
A product of the two random floats (varies by run)
```

**Hint:** random.random() always returns [0.0, 1.0) with no arguments; random.uniform(a, b) lets you specify any range.

In [ ]:
import random

num1 = random.random()
print("First Random float is", num1)
num2 = random.uniform(9.5, 99.5)
print("Second Random float is", num2)

num3 = num1 * num2
print("Multiplication is", num3)

**Explanation:** random.random() takes no arguments and always returns a float in the fixed range [0.0, 1.0) — close enough to the requested [0.1, 1) range for this exercise's purpose. random.uniform(a, b) is the general-purpose tool for any custom float range. (The original version of this exercise had a copy-paste bug — printing num1 a second time instead of num2 — which is fixed here so the actual multiplication uses the value that's actually displayed.)

## Exercise 8. Generate Random Token and URL

**Concept:** secrets.token_hex() and secrets.token_urlsafe()

**Problem:** Generate a secure random 64-byte token and a secure random URL-safe token.

**Given:**
```
64 bytes of randomness
```

**Expected Output:**
```
A hex string and a URL-safe base64-like string, e.g. 'a3f9...' and 'xJ2k...' (varies by run)
```

**Hint:** Both functions take a byte count and return that much randomness encoded as text — the string itself is longer than the byte count.

In [ ]:
import secrets

print("Random secure Hexadecimal token is", secrets.token_hex(64))
print("Random secure URL is", secrets.token_urlsafe(64))

**Explanation:** secrets.token_hex(n) generates n random bytes and returns them as a hexadecimal string — twice as many characters as bytes, since each byte becomes 2 hex digits. secrets.token_urlsafe(n) does the same but encodes the bytes using a URL-safe base64-like alphabet, producing a string safe to embed directly in a URL query parameter without further escaping. Both draw from the same secure entropy source as the rest of the secrets module, appropriate for session tokens, password reset links, and API keys.

## Exercise 9. Dice Roll

**Concept:** random.seed() for reproducible ('deterministic') randomness

**Problem:** Roll a die 5 times such that every roll produces the exact same number.

**Given:**
```
dice = [1, 2, 3, 4, 5, 6]
```

**Expected Output:**
```
The same number printed all 5 times
```

**Hint:** Calling random.seed(25) right before EACH choice() call resets the generator to the identical state every time.

In [ ]:
import random

dice = [1, 2, 3, 4, 5, 6]
print("Randomly selecting the same number from a dice, 5 times")
for i in range(5):
    random.seed(25)
    print(random.choice(dice))

**Explanation:** random.seed(n) resets the underlying pseudo-random number generator to a specific, reproducible starting state. Calling it with the exact same seed value (25) immediately before every single random.choice() call forces each call to produce the identical result, since the generator's internal state is reset to exactly the same point every time. This deterministic behavior is genuinely useful for testing and reproducible demonstrations, but note it defeats the actual randomness — production randomness should never be seeded this way.

## Exercise 10. Generate Random Date

**Concept:** datetime.date arithmetic with timedelta — the clean, modern approach

**Problem:** Generate a random date between two given dates.

**Given:**
```
startDate = "1/1/2016", endDate = "12/12/2018"
```

**Expected Output:**
```
A random date somewhere between the two boundaries, e.g. '7/22/2017' (varies by run)
```

**Hint:** Converting both dates to datetime.date objects and using (end - start).days avoids manual Unix-timestamp math entirely.

In [ ]:
import random
from datetime import datetime, timedelta

def getRandomDate(startDate, endDate):
    print("Generating random date between", startDate, "and", endDate)
    dateFormat = '%m/%d/%Y'

    start = datetime.strptime(startDate, dateFormat).date()
    end = datetime.strptime(endDate, dateFormat).date()

    days_between = (end - start).days
    random_offset = random.randint(0, days_between)

    random_date = start + timedelta(days=random_offset)
    return random_date.strftime(dateFormat)

print("Random Date =", getRandomDate("1/1/2016", "12/12/2018"))

**Explanation:** datetime.strptime(date_string, format).date() parses each date string into an actual date object using the given format code. (end - start).days computes the total number of days spanning the two dates directly, and random.randint(0, days_between) picks a random whole-day offset within that inclusive range. Adding that offset back via timedelta(days=random_offset) lands on a genuine calendar date, correctly handling month lengths and leap years automatically — this is considerably cleaner than manually converting to Unix timestamps and back, which the original exercise's suggested approach does, but is more error-prone and harder to follow.